# Router Agent

In this notebook we build a router agent that, depending on the user query, will answer differently. We use LangGraph to create a graph of how the model should behave.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/03-mcp-router.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Understand how a router agent directs queries to different personas
- Build a router agent using LangGraph with conditional edges
- Integrate MCP tools to provide real-time data within an agent
- Interact with the agent through a notebook dashboard

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [ ]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Build Agent

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
import re
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from IPython.display import display
import ipywidgets as widgets

### Launch MCP Server

We are going to expose an MCP server that can provide current weather information.

In [ ]:
server_params = StdioServerParameters(command="python", args=['open_weather.py'])

### Create Agent State

To manage our agent state, we create an object that can keep track of the messages. The `operator.add` makes sure that new messages are appended and not erased.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

### Create Router Agent

We create the `Agent` class.

The `__init__` function initializes the `StateGraph` and defines the different nodes and edges of the graph. Note the conditional edge that depends on `self.router_decision`. Once the graph is fully defined we can compile it. We also initialize the model with a local Ollama endpoint.

`self.router` takes the user query and uses a system prompt to get the LLM to provide the transition to the next state, which will either provide an answer as a scientist or poet.

`self.router_decision` simply gets the message state and returns the LLM response.

`self.play_writer` is called only when the agent identifies that the response should be in the form of a play. We use the system prompt to guide the model in the response. We also use the weather MCP tools to get current weather conditions in case a place is part of the query.

`scientist_writer` simply uses the system prompt to answer as a scientist.

In [ ]:
class Agent:
    def __init__(self, model: str="qwen3.5:9b"):
        graph = StateGraph(AgentState)
        graph.add_node("router", self.router)
        graph.add_node("play_writer", self.play_writer)
        graph.add_node("scientist_writer", self.scientist_writer)
        graph.add_edge(START, "router")
        graph.add_edge("play_writer", END)
        graph.add_edge("scientist_writer", END)
        graph.add_conditional_edges(
            "router",
            self.router_decision,
            {"play": "play_writer", "scientist": "scientist_writer"}
        )
        self.graph = graph.compile()
        self.model = ChatOpenAI(model=model, base_url="http://localhost:11434/v1/", api_key="abc-123")


    def router(self, state: AgentState):
        prompt = """You are a useful agent that takes the user query and based on the analysis of the query
        you return 'play' when a play writer is more applicable or 'scientist' when a scientific response is
        more applicable. Do not explain your reasoning, only return 'play' or 'scientist' always in lowercase.
        """
        user_query = state['messages']

        messages = [SystemMessage(content=prompt)] + user_query

        message = self.model.invoke(messages)
        return {'messages': [message]}

    def router_decision(self, state: AgentState):
        """Returns 'play' or 'scientist' """
        result = state['messages'][-1]
        content = result.content.lower()
        # Strip <think></think> tags if present
        content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()
        return content

    async def play_writer(self, state: AgentState):
        user_query = state['messages'][-2]

        prompt = """You are a useful agent that writes plays in the style of William Shakespeare.
        Based on the user query provide a compelling yet short play.

        If you are provided with a place, pick a random location to get the current weather that
        should be part of the play. If you are provided with weather conditions use those as part of the poem.

        Sign the play with a reference to Shakespeare.
        """

        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                model_with_tools = self.model.bind_tools(tools)
                messages = [SystemMessage(content=prompt)] + [user_query]

                message = await model_with_tools.ainvoke(messages)
                messages.append(message)

                if message.tool_calls:
                    for tool_call in message.tool_calls:
                        tool_call_result = await session.call_tool(tool_call['name'], arguments=tool_call['args'])
                        tool_message = ToolMessage(
                            content=tool_call_result.content[-1].text,
                            tool_call_id=tool_call['id'],
                        )
                        messages.append(tool_message)

                    message = await model_with_tools.ainvoke(messages)
                    messages.append(message)

                return {'messages': messages[1:]}

    def scientist_writer(self, state: AgentState):
        user_query = state['messages'][-2]
        prompt = """You are a useful agent that writes as if you were the scientist Stephen Hawking.
        Based on the user query provide a compelling yet short description.

        Sign the description with a reference to A Brief History of Time.
        """

        messages = [SystemMessage(content=prompt)] + [user_query]

        message = self.model.invoke(messages)
        return {'messages': [message]}

Let us create an instance of our agent

In [ ]:
writer_agent = Agent(model="qwen3.5:9b")

### Visualize the Agent Graph

We can now visualize what our graph looks like.

In [ ]:
display({"text/vnd.mermaid": writer_agent.graph.get_graph().draw_mermaid()}, raw=True)

### Use the Agent

Let us query the agent and try to get a story about the Mediterranean

In [ ]:
messages = [HumanMessage(content="Tell me a story about the Mediterranean.")]
response_mediterranean = await writer_agent.graph.ainvoke({"messages": messages})

Let us show the response of our agent

In [ ]:
response_mediterranean['messages'][-1].pretty_print()

You can also explore the full agent message history

In [ ]:
for history in response_mediterranean['messages']:
    history.pretty_print()

Let us query the agent and try to get a scientific response

In [ ]:
messages = [HumanMessage(content="Tell me why the sky is blue")]
response_sky = writer_agent.graph.invoke({"messages": messages})

In [ ]:
response_sky['messages'][-1].pretty_print()

## Agent Dashboard

The dashboard below lets you interact with the router agent directly from the notebook.

- **Type a prompt** in the text area and click **Run Agent** to send it to the agent.
- The button text changes to **Running…** while the agent processes your request.
- Once complete, the agent's **final response** is displayed automatically in the output box.
- Use the **Message slider** to browse through all intermediate messages from the latest run (e.g., the router decision, tool calls, and the final response).
- Each new run **replaces** the previous results, so you always see messages from the most recent query.

In [ ]:
latest_messages = []
prompt_area = widgets.Textarea(placeholder="Type your prompt here…", layout=widgets.Layout(width="100%", height="80px"))
run_button = widgets.Button(description="Run Agent", button_style="primary", layout=widgets.Layout(width="auto"))
slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Message Index:", continuous_update=False, layout=widgets.Layout(width="50%"), style={"description_width": "initial"})
output_box = widgets.Textarea(value="", disabled=True, layout=widgets.Layout(width="100%", height="200px"))

def on_slider_change(change):
    idx = change["new"]
    if latest_messages:
        msg = latest_messages[idx]
        output_box.value = msg.pretty_repr()

async def on_run_clicked(b):
    global latest_messages

    user_text = prompt_area.value.strip()
    if not user_text:
        return

    latest_messages = []
    prompt_area.value = ""
    output_box.value = ""
    slider.value = 0
    slider.max = 0
    run_button.description = "Running…"
    run_button.disabled = True

    try:
        result = await writer_agent.graph.ainvoke(
            {"messages": [HumanMessage(content=user_text)]}
        )
        latest_messages = result.get("messages", [])
        if latest_messages:
            slider.max = len(latest_messages) - 1
            slider.value = len(latest_messages) - 1
            output_box.value = latest_messages[-1].pretty_repr()
        else:
            output_box.value = "No messages returned"
    except Exception as exc:
        output_box.value = str(exc)
    finally:
        run_button.description = "Run Agent"
        run_button.disabled = False

slider.observe(on_slider_change, names="value")
run_button.on_click(lambda b: __import__("asyncio").ensure_future(on_run_clicked(b)))
dashboard = widgets.VBox(
    [
        prompt_area,
        widgets.HBox([run_button], layout=widgets.Layout(justify_content="center")),
        slider,
        output_box,
    ],
    layout=widgets.Layout(max_width="700px", overflow="hidden", padding="0 4px 0 0"),
)
display(dashboard)

## Exercises for the Reader

- Use a Reasoning model such as qwen3.5
  - Do you notice any difference in the agent answers?
- Add another path to the router where you add a different persona, e.g., historian, engineer, etc.

## Conclusions

In this notebook we built a router agent using LangGraph that directs user queries to different personas based on the content of the query. We used conditional edges in the state graph to route between a play writer and a scientist writer. We also integrated MCP tools to enrich the play writer with real-time weather data.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://docs.langchain.com/oss/python/langgraph/overview">LangGraph</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT